# Test the deposited processed files, without Sherlock input data

Place the two downloaded H5MU files in `<repository>/geo_release/`, or set `CCPERTURBSEQ_GEO_DIR` to their local folder before starting Jupyter. Only the processed files and this repository's code/environment are required. This notebook does **not** read `config/paths.local.yaml`, source experiment folders, original H5ADs, external spectra, or external saturation reports.

Tests include unique/aligned cell IDs, finite nonnegative counts, stored matrix checksums, Hippo normalized-usage/CLR consistency, original QC/panel masks, independent reconstruction of S2 guide metrics from its sparse guide matrix, and equality of regenerated panel tables with the embedded pre-export references. Original panel producers export PDF/SVG/PNG previews for visual comparison.

A passing result establishes roundtrip and panel-input equivalence for these files. It is not a claim that GEO metadata, FASTQ mapping, or the manuscript's other experiments have been validated.

In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "src" / "cc_perturbseq").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the manuscript-analysis repository checkout.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from cc_perturbseq.geo_submission import (
    HIPPO_FILENAME, MIX_FILENAME, validate_submission, file_checksums,
)
RELEASE_DIR = Path(os.environ.get("CCPERTURBSEQ_GEO_DIR", str(REPO_ROOT / "geo_release"))).expanduser().resolve()
print("Processed-file directory:", RELEASE_DIR)


In [ ]:
TEST_HIPPO = True
TEST_MIX = True
RENDER_PANELS = True
TEST_OUTPUT = RELEASE_DIR / "public_reproduction_test"
TEMP_DIR = TEST_OUTPUT / "_temporary"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
for enabled, name in ((TEST_HIPPO, HIPPO_FILENAME), (TEST_MIX, MIX_FILENAME)):
    if enabled and not (RELEASE_DIR / name).is_file():
        raise FileNotFoundError(RELEASE_DIR / name)


In [ ]:
reports = {}
if TEST_HIPPO:
    reports["hippo"] = validate_submission(
        RELEASE_DIR / HIPPO_FILENAME,
        render_dir=TEST_OUTPUT / "hippo" if RENDER_PANELS else None,
        temp_parent=TEMP_DIR,
    )
    display(reports["hippo"])


In [ ]:
if TEST_MIX:
    reports["mixed_cells"] = validate_submission(
        RELEASE_DIR / MIX_FILENAME,
        render_dir=TEST_OUTPUT / "mixed_cells" if RENDER_PANELS else None,
    )
    display(reports["mixed_cells"])


In [ ]:
assert reports and all(report["status"] == "PASS" for report in reports.values())
(TEST_OUTPUT / "all_validation_reports.json").write_text(json.dumps(reports, indent=2) + "\n")
print("All selected processed files passed numerical reproduction checks.")
print("Generated figures and source tables:", TEST_OUTPUT)
if RENDER_PANELS:
    from IPython.display import Image
    for path in sorted(TEST_OUTPUT.rglob("*.png")):
        print(path.name)
        display(Image(filename=str(path)))


## Inspect either object directly

```python
import mudata as md
hippo = md.read_h5mu(RELEASE_DIR / "Hippo_timecourse_integrated.h5mu")
rna = hippo.mod["rna"]
programs = hippo.mod["cnmf"]
# rna.X: raw RNA counts; programs.X: normalized cNMF usage.
# programs.obsm["X_cnmf_clr_top_specific_200"]: frozen CLR matrix.

mixed = md.read_h5mu(RELEASE_DIR / "iPSC_K562_mixed_integrated.h5mu")
rna = mixed.mod["rna"]
guides = mixed.mod["crispr"]
# guides.X: sparse original guide counts, before the panel-n threshold.
# rna.obs: sample/lineage annotations and all per-cell panel metrics.
# mixed.uns["saturation"]: all four embedded saturation curves.
```

For memory-limited inspection, read one object at a time. Input-file paths inside provenance or inherited metadata are historical records only; the reproduction functions do not open them.